# 06. Amazon Bedrock Intelligent Prompt Routing 체험

지금까지의 실습(SGLang decision layer)이 **"판단용 LLM 호출을 아예 제거"** 하는 접근이라면, Bedrock **Intelligent Prompt Routing(IPR)** 은 **"생성 호출이 어느 모델로 갈지를 요청마다 골라주는"** 관리형 기능입니다.

| 비교 | Decision layer (01-05 실습) | Intelligent Prompt Routing |
|---|---|---|
| 하는 일 | 선택지 중 label 하나를 즉답 (prefill-only) | 요청 난이도를 예측해 같은 family의 작은/큰 모델 중 선택 |
| 대상 | agent 내부의 판단 노드 | 고객 응답을 생성하는 LLM 호출 자체 |
| 효과 | 판단 latency 수십 ms, 판단 비용 ~0 | 쉬운 요청을 작은 모델로 보내 생성 비용 절감 |
| 관계 | **보완재** — 판단은 decision layer로 제거하고, 남은 생성 호출은 IPR로 라우팅할 수 있습니다 | |

호출 방법은 간단합니다: `converse()`의 `modelId` 자리에 **router ARN**을 넣으면, 응답의 `trace.promptRouter.invokedModelId`로 어느 모델이 선택됐는지 알 수 있습니다.

## 1. 이 리전의 prompt router 목록

us-west-2에는 family별 **default router** 3종이 제공되고, `CreatePromptRouter`로 **custom router**도 만들 수 있습니다.

> **실습 모델 안내 (2026-10 기준)**: default Anthropic router는 Claude 3 Haiku / 3.5 Sonnet을 멤버로 참조하는데 이 모델들은 **EOL**이어서 호출이 거부되고, 최신 Claude 4.5 계열은 아직 prompt routing **미지원**입니다(CreatePromptRouter가 ValidationException 반환 — 직접 확인). 따라서 이 노트북은 **Nova family(Lite/Pro)** 로 라이브 실습합니다. 동작 원리와 API는 family와 무관하게 동일합니다.

In [1]:
import boto3, json, os, time, statistics
from IPython.display import Markdown, display

# region/account는 실행 환경에서 동적으로 결정 (workshop notebook instance = us-east-1)
REGION = os.environ.get("BEDROCK_REGION") or boto3.session.Session().region_name or "us-east-1"
ACCT = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
br = boto3.client("bedrock", region_name=REGION)
rt = boto3.client("bedrock-runtime", region_name=REGION)
print(f"region={REGION} account={ACCT}")

rows = ["| Router | type | qualityDiff | 멤버 모델 |", "|---|---|---|---|"]
for r in br.list_prompt_routers(type="custom")["promptRouterSummaries"]:
    models = ", ".join(m["modelArn"].split("/")[-1] for m in r["models"])
    rows.append(f"| {r['promptRouterName']} | {r['type']} | "
                f"{r['routingCriteria']['responseQualityDifference']} | {models} |")
display(Markdown("\n".join(rows)))

region=us-east-1 account=051934905721


| Router | type | qualityDiff | 멤버 모델 |
|---|---|---|---|
| ws-nova-router | custom | 25.0 | us.amazon.nova-lite-v1:0, us.amazon.nova-pro-v1:0 |

## 2. 같은 상담 질문을 router로 보내 보기

01-04 실습에서 썼던 한국어 상담 질문들을 **Nova default router**에 그대로 보내고, 요청마다 **어느 모델이 선택되는지**와 latency를 관찰합니다. 쉬운 인사/FAQ는 Lite로, 복잡한 송금 문의는 Pro로 가는지 확인해 보세요.

In [2]:
DEFAULT_NOVA = f"arn:aws:bedrock:{REGION}:{ACCT}:default-prompt-router/amazon.nova:1"

QUESTIONS = [
    ("인사/FAQ (쉬움)", "안녕하세요? 영업시간이 어떻게 되나요?"),
    ("카드 문의 (중간)", "카드가 안 긁히는데 어떻게 해야 하나요? 간단히 답해줘."),
    ("송금 문의 (어려움)", "해외 송금이 이틀째 도착하지 않았습니다. SWIFT 중개은행 수수료 차감, "
                        "수취은행 컷오프, AML 보류 각각의 확인 절차와 고객 안내 문구를 정리해 주세요."),
]

def route_once(router_arn, text, max_tokens=200):
    t0 = time.perf_counter()
    r = rt.converse(modelId=router_arn,
                    messages=[{"role": "user", "content": [{"text": text}]}],
                    inferenceConfig={"maxTokens": max_tokens})
    ms = (time.perf_counter() - t0) * 1000
    picked = r.get("trace", {}).get("promptRouter", {}).get("invokedModelId", "?").split("/")[-1]
    answer = r["output"]["message"]["content"][0]["text"]
    return picked, ms, answer, r["usage"]

default_available = True
for tag, q in QUESTIONS:
    print(f"[{tag}]")
    print(f"  고객 질문: {q}")
    try:
        picked, ms, answer, usage = route_once(DEFAULT_NOVA, q)
    except Exception as e:
        default_available = False
        print("  (default router 사용 불가 — cross-region 호출이 이 계정에서 차단됨.")
        print("   default router는 us-east-1/us-east-2/us-west-2로 팬아웃하므로 허용 리전이")
        print("   잠긴 계정(SCP)에서 발생합니다. 다음 셀의 custom router로 진행하세요.)")
        print(f"   {type(e).__name__}: {str(e)[:120]}")
        break
    print(f"  선택 모델: {picked}   ({ms:,.0f}ms, in {usage['inputTokens']}/out {usage['outputTokens']} tok)")
    print(f"  답변: {answer.strip()[:160]}")
    print()


[인사/FAQ (쉬움)]
  고객 질문: 안녕하세요? 영업시간이 어떻게 되나요?


  선택 모델: us.amazon.nova-lite-v1:0   (1,685ms, in 26/out 200 tok)
  답변: 안녕하세요! 저는 인공지능 어시스턴트이고, 항상 24시간 온라인으로 도움을 드릴 수 있습니다. 만약 특정 업체나 상점의 영업시간에 대한 정보가 필요하시다면, 해당 업체의 웹사이트를 방문하거나 전화로 문의해 보시는 것이 좋습니다. 혹시 특정 업체의 영업시간에 대한 정보가 필요하시다면 알려

[카드 문의 (중간)]
  고객 질문: 카드가 안 긁히는데 어떻게 해야 하나요? 간단히 답해줘.


  선택 모델: us.amazon.nova-pro-v1:0   (1,463ms, in 40/out 200 tok)
  답변: 카드가 안 긁히도록 하려면 다음과 같은 방법을 시도해 보세요:

1. **카드 케이스 사용**: 카드를 보호하기 위한 케이스나 홀더를 사용하세요.
2. **지갑 선택**: 카드를 긁힘으로부터 보호할 수 있는 튼튼한 지갑을 선택하세요.
3. **분리 보관**: 카드를 열쇠나 동전과 같이 

[송금 문의 (어려움)]
  고객 질문: 해외 송금이 이틀째 도착하지 않았습니다. SWIFT 중개은행 수수료 차감, 수취은행 컷오프, AML 보류 각각의 확인 절차와 고객 안내 문구를 정리해 주세요.


  선택 모델: us.amazon.nova-pro-v1:0   (1,829ms, in 100/out 200 tok)
  답변: 물론입니다. 해외 송금이 지연될 때 발생할 수 있는 주요 이유와 고객에게 안내할 수 있는 문구를 정리해 드리겠습니다.

### 1. SWIFT 중개은행 수수료 차감

**확인 절차:**
1. 송금 명세서에서 중개은행(SWIFT 경유 은행)을 확인합니다.
2. 해당 은행의 수수료 정책을 



## 3. Custom router 만들기 — 라우팅 성향 조절

`routingCriteria.responseQualityDifference`(%, 5의 배수)는 "작은 모델의 예상 품질이 큰 모델보다 이만큼 낮아도 작은 모델을 쓰겠다"는 허용폭입니다. 값이 클수록 작은 모델로 더 공격적으로 라우팅됩니다.

아래 custom router(허용폭 25%)는 이 계정에 이미 생성되어 있습니다. 생성 코드는 다음과 같습니다(재실행 불필요):

```python
br.create_prompt_router(
    promptRouterName="ws-nova-router",
    routingCriteria={"responseQualityDifference": 25},   # %, 5의 배수만 허용
    models=[{"modelArn": ".../inference-profile/us.amazon.nova-lite-v1:0"},
            {"modelArn": ".../inference-profile/us.amazon.nova-pro-v1:0"}],
    fallbackModel={"modelArn": ".../inference-profile/us.amazon.nova-pro-v1:0"})
```

In [3]:
# custom router: 이 계정에 없으면 생성 (get-or-create)
CUSTOM_NAME = "ws-nova-router"
found = [r for r in br.list_prompt_routers(type="custom")["promptRouterSummaries"]
         if r["promptRouterName"] == CUSTOM_NAME]
if found:
    CUSTOM_NOVA = found[0]["promptRouterArn"]
else:
    CUSTOM_NOVA = br.create_prompt_router(
        promptRouterName=CUSTOM_NAME,
        routingCriteria={"responseQualityDifference": 25},
        # 멤버를 리전 내 foundation model로 지정 — 호출이 이 리전 안에서 완결되어
        # 리전 잠금(SCP) 계정에서도 동작한다 (us. inference profile은 cross-region 팬아웃)
        models=[{"modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-lite-v1:0"},
                {"modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-pro-v1:0"}],
        fallbackModel={"modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-pro-v1:0"},
    )["promptRouterArn"]
print("custom router:", CUSTOM_NOVA)


rows = ["| 질문 | default (diff=0) | custom (diff=25) |", "|---|---|---|"]
for tag, q in QUESTIONS:
    p2, m2, _, _ = route_once(CUSTOM_NOVA, q)
    if default_available:
        p1, m1, _, _ = route_once(DEFAULT_NOVA, q)
        left = f"{p1.replace('us.amazon.','').replace('amazon.','')} ({m1:,.0f}ms)"
    else:
        left = "(cross-region 차단)"
    rows.append(f"| {tag} | {left} "
                f"| {p2.replace('us.amazon.','').replace('amazon.','')} ({m2:,.0f}ms) |")
display(Markdown("\n".join(rows)))

custom router: arn:aws:bedrock:us-east-1:051934905721:prompt-router/ol541pgvb39r


| 질문 | default (diff=0) | custom (diff=25) |
|---|---|---|
| 인사/FAQ (쉬움) | nova-lite-v1:0 (1,571ms) | nova-lite-v1:0 (1,257ms) |
| 카드 문의 (중간) | nova-pro-v1:0 (1,635ms) | nova-lite-v1:0 (1,527ms) |
| 송금 문의 (어려움) | nova-pro-v1:0 (1,798ms) | nova-pro-v1:0 (1,817ms) |

## 4. Decision layer와 어떻게 함께 쓰나

- **판단(D1-D8)**: IPR로는 해결되지 않습니다 — 어느 모델로 라우팅하든 판단 1회는 LLM 생성 호출 1회(수백 ms, 과금)입니다. 판단은 decision layer(수십 ms, 비용 ~0)가 맞습니다.
- **생성(respond, summarize)**: 여기가 IPR의 자리입니다. 쉬운 상담 응답은 작은 모델이 쓰게 하여 생성 비용을 줄입니다.
- 조합 예: `guardrail/routing/escalation = SGLang decision` + `최종 응답 생성 = IPR(작은/큰 모델 자동 선택)` — 판단 latency와 생성 비용을 동시에 줄이는 구성입니다.

**유의점**
- router의 난이도 예측 자체도 완벽하지 않으므로, 품질 민감 업무는 responseQualityDifference를 보수적으로(0-10) 두고 시작하세요.
- IPR은 **같은 family 안에서만** 라우팅합니다 (Claude<->Nova 혼합 불가).
- latency는 선택된 모델의 생성 시간이 지배합니다 — IPR은 비용 최적화 도구이지, decision layer 같은 latency 제거 도구가 아닙니다.
- 과금: IPR 자체 요금 **$1 / 1,000 requests** + 선택된 모델의 토큰 요금 (aws.amazon.com/bedrock/pricing). 판단 1회당 $0.001 고정비가 추가되는 셈이라, 수십만 건 이상의 '판단' 용도로는 decision layer(한계비용 ~0)와 비용 구조가 근본적으로 다릅니다.